In [16]:
import pickle
import librosa
import numpy as np

clf    = pickle.load(open('models/model.pkl', 'rb'))
scaler = pickle.load(open('models/scaler.pkl', 'rb'))
THRESHOLD = 0.7

def predict(wav_path):
    audio, _ = librosa.load(wav_path, sr=16000, mono=True)
    mfcc = librosa.feature.mfcc(y=audio, sr=16000, n_mfcc=40)
    feat = np.mean(mfcc.T, axis=0).reshape(1, -1)
    feat_s = scaler.transform(feat)
    prob = clf.predict_proba(feat_s)[0][1]
    detected = prob >= THRESHOLD
    print(f'File  : {wav_path}')
    print(f'Score : {prob:.4f}')
    print(f'Kết quả: {"✅ HEY SIRI DETECTED!" if detected else "❌ Không phải wake word"}')
    print()

# Test vài file positive
print('=== Test Positive ===')
for wav in list(Path('data/positive').glob('*.wav'))[:3]:
    predict(str(wav))

# Test vài file negative  
print('=== Test Negative ===')
for wav in list(Path('data/negative').glob('*.wav'))[:3]:
    predict(str(wav))

=== Test Positive ===
File  : data/positive/pos_009.wav
Score : 1.0000
Kết quả: ✅ HEY SIRI DETECTED!

File  : data/positive/pos_035.wav
Score : 1.0000
Kết quả: ✅ HEY SIRI DETECTED!

File  : data/positive/pos_021.wav
Score : 1.0000
Kết quả: ✅ HEY SIRI DETECTED!

=== Test Negative ===
File  : data/negative/neg_022.wav
Score : 0.0000
Kết quả: ❌ Không phải wake word

File  : data/negative/neg_036.wav
Score : 0.0100
Kết quả: ❌ Không phải wake word

File  : data/negative/neg_037.wav
Score : 0.0200
Kết quả: ❌ Không phải wake word



In [17]:
import sounddevice as sd

DURATION = 3  # giây

print(f'🎤 Đang nghe {DURATION} giây... Nói "Hey Siri"!')
audio = sd.rec(int(DURATION * 16000), samplerate=16000, channels=1, dtype='float32')
sd.wait()
audio = audio.flatten()

mfcc = librosa.feature.mfcc(y=audio, sr=16000, n_mfcc=40)
feat = np.mean(mfcc.T, axis=0).reshape(1, -1)
feat_s = scaler.transform(feat)
prob = clf.predict_proba(feat_s)[0][1]
detected = prob >= THRESHOLD

print(f'Score  : {prob:.4f}')
print(f'Kết quả: {"✅ HEY SIRI DETECTED!" if detected else "❌ Không phải wake word"}')

🎤 Đang nghe 3 giây... Nói "Hey Siri"!
Score  : 0.5900
Kết quả: ❌ Không phải wake word
